# awseasy

> AWS cloud provisioning for GenAI workloads — made easy.

## Install

```sh
pip install awseasy
```

## Overview

`awseasy` is the AWS sibling of [`azeasy`](https://github.com/vedicreader/azeasy). Same patterns — thin Pythonic wrappers over AWS SDKs, nbdev + fastcore, compliance as composable dict profiles.

| Tool | Purpose |
|---|---|
| dockeasy | Docker/Compose config generation |
| cfeasy | Cloudflare DNS + Zero Trust tunnels |
| vpseasy | Hetzner VPS provisioning + deployment |
| fastops | DevOps toolkit — containers, reverse proxies, VMs |
| azeasy | Azure cloud provisioning for GenAI workloads |
| **awseasy** | **AWS cloud provisioning for GenAI workloads** |

## Authentication

`awseasy` uses the standard boto3 credential chain — no hardcoded secrets:

1. Environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`)
2. AWS profile (`~/.aws/credentials`)
3. EC2 / ECS instance profile (IAM role)
4. EKS pod identity / IRSA
5. AWS SSO via IAM Identity Center

```python
from awseasy import *

auth = AWSAuth()                         # uses AWS_DEFAULT_REGION
auth = AWSAuth(region='eu-west-1')       # explicit region
auth = AWSAuth(profile='staging')        # named profile
auth = AWSAuth(role_arn='arn:aws:...')   # cross-account assume-role
```

## One-call GenAI stack

```python
from awseasy import *

auth = AWSAuth()
stack = GenAIStack(auth, 'myapp', compliance=HIPAA)
stack.provision()   # IAM role, Secrets Manager, S3, OpenSearch, Bedrock KB, DynamoDB, Redis
print(stack.summary())
```

## Modules

| Module | Services |
|---|---|
| `awseasy.core` | `AWSAuth`, compliance profiles, resource groups, `GenAIStack` |
| `awseasy.ai` | Amazon Bedrock, Bedrock Knowledge Bases, OpenSearch |
| `awseasy.data` | S3, DynamoDB, RDS PostgreSQL, ElastiCache Redis |
| `awseasy.compute` | EC2, EKS, ECR |
| `awseasy.network` | VPC/SG, Secrets Manager, IAM, VPC Endpoints, CloudFront, ALB |

## Compliance profiles

```python
create_bucket(auth, 'phi-data', **HIPAA)       # encryption, no public access, 35-day backups
create_postgres(auth, 'app-db', **ISO27001)    # encryption, least-privilege role
create_redis(auth, 'cache', **SOC2)            # encryption, 7-day backups
```

Every `create_*` function accepts `**compliance_opts` — compliance requirements compose naturally.

## Security defaults

- **Encryption at rest** — all storage services (S3, DynamoDB, RDS, Redis, ECR) encrypted by default
- **Encryption in transit** — TLS enforced on all endpoints
- **IMDSv2** — EC2 instances enforce token-based metadata access
- **No public access** — S3 public access blocked; RDS/Redis created without public routes
- **IAM roles** — prefer roles over long-lived access keys; `create_role()` + `attach_policy()`
- **Secrets Manager** — store credentials with `create_secret()`, never hardcode
- **Idempotency** — all `create_*` functions are safe to run multiple times